# Section 2 Results

Run `python src/section2.py` first. This notebook displays the data, portfolio weights, statistics, and figures required by each question.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

project_root = Path.cwd()
if not (project_root / 'src' / 'section2.py').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))
from section2 import DECISION_DATE, LOOKBACK_START, REALIZED_END, RETURN_PENALTY

section2_dir = project_root / 'data' / 'section2'
daily_returns_path = project_root / 'data' / 'daily_returns.csv'

daily_returns = pd.read_csv(daily_returns_path, index_col='Date', parse_dates=['Date'])
assignment_returns = daily_returns.loc[LOOKBACK_START:REALIZED_END]
inputs = pd.read_csv(section2_dir / 'section2_estimation_inputs.csv')
weights = pd.read_csv(section2_dir / 'section2_weights.csv')
metrics = pd.read_csv(section2_dir / 'section2_metrics.csv')
realized = pd.read_csv(section2_dir / 'section2_realized_returns.csv', index_col='Date', parse_dates=['Date'])
dynamic_comparison = pd.read_csv(section2_dir / 'section2_dynamic_comparison.csv', index_col='Metric')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.6f}'.format)

## Preliminary data

The cleaned daily-return data used by the analysis.

In [ ]:
print(f'Assignment data window: {assignment_returns.index.min().date()} to {assignment_returns.index.max().date()}')
print(f'Observations: {len(assignment_returns)}')
print(f'Assets: {list(assignment_returns.columns)}')
display(assignment_returns.head())
display(assignment_returns.isna().sum().rename('missing_values').to_frame().T)

## 1. Two-fund Theorem with Target Beta

In [ ]:
expected_returns = inputs[inputs['input_type'] == 'expected_return'].set_index('asset')['value'].rename('Expected Return')
betas = inputs[inputs['input_type'] == 'beta'].set_index('asset')['value'].rename('Beta')
covariance = inputs[inputs['input_type'] == 'covariance'].pivot(index='asset', columns='other_asset', values='value')
estimation_summary = pd.concat([expected_returns, betas], axis=1)
display(estimation_summary.round(6))

print('Sample covariance matrix:')
display(covariance.round(6))

q1_names = ['q1_target_beta_0.5', 'q1_target_beta_1.5', 'q1_theorem_target_beta_1', 'q1_direct_target_beta_1']
q1_weights = weights[weights['portfolio'].isin(q1_names)].pivot(index='asset', columns='portfolio', values='weight')
display(q1_weights.round(6))
q1_result = metrics.loc[metrics['portfolio'] == 'two_fund_alpha', ['alpha', 'theorem_direct_max_weight_difference']].T
q1_result.columns = ['value']
display(q1_result.round(12))

## 2(a). Betas relative to the S&P-500

The table includes the individual betas for the investment universe and AAPL.

In [ ]:
display(betas.to_frame().round(6))

## 2(b). Beta Hedging Portfolio

In [ ]:
q2b_names = ['q2b_beta_hedge_basket', 'q2b_aggregated_hedged_position']
q2b_weights = weights[weights['portfolio'].isin(q2b_names)].pivot(index='asset', columns='portfolio', values='weight')
display(q2b_weights.round(6))

q2b_metrics = metrics[metrics['portfolio'].isin(q2b_names)].set_index('portfolio')
display(q2b_metrics[['expected_return_daily', 'expected_return_annualized', 'volatility_daily', 'ex_ante_beta', 'weight_sum']].round(6))
print('The hedge strategy is long AAPL and short the beta-hedging basket.')

## 2(c). Beta Neutral Portfolio

In [ ]:
q2c_names = ['q2c_beta_neutral_portfolio']
q2c_weights = weights[weights['portfolio'].isin(q2c_names)].pivot(index='asset', columns='portfolio', values='weight')
display(q2c_weights.round(6))

q2c_comparison = metrics[metrics['portfolio'].isin(['aggregated_hedged_position', 'beta_neutral_portfolio'])].set_index('portfolio')
display(q2c_comparison[['expected_return_daily', 'volatility_daily', 'ex_ante_beta', 'realized_beta']].round(6))

## 3(a). Realized daily returns

In [ ]:
print(f'Realized period: {realized.index.min().date()} to {realized.index.max().date()}')
print(f'Observations: {len(realized)}')
display(realized[['beta_hedged_strategy', 'beta_neutral_portfolio']].head(10).round(6))

## 3(b). Comparison of the two hedging strategies

The VaR and CVaR columns are reported as losses. A small negative VaR means the empirical 5th-percentile return is slightly positive in this sample.

In [ ]:
q3_names = ['q3_beta_hedged_strategy', 'q3_beta_neutral_portfolio']
q3_metrics = metrics[metrics['portfolio'].isin(q3_names)].set_index('portfolio')
q3_fields = ['realized_mean_daily', 'realized_volatility_daily', 'sharpe_annualized', 'sortino_annualized', 'var_95_loss', 'cvar_95_loss', 'skewness', 'kurtosis_excess', 'realized_beta', 'realized_cumulative_return']
display(q3_metrics[q3_fields].round(6))
display(Image(filename=str(section2_dir / 'section2_comparison.png')))

## 4(a). Chasing Performance

In [ ]:
q4a_ex_ante_names = ['q2b_aggregated_hedged_position', 'q2.4a_aggregated_hedged_position']
print(f'Lambda used in the return-term objective: {RETURN_PENALTY}')
q4a_weight_names = ['q2.4a_beta_hedge_lambda_2', 'q2.4a_aggregated_hedged_position']
q4a_weights = weights[weights['portfolio'].isin(q4a_weight_names)].pivot(index='asset', columns='portfolio', values='weight')
display(q4a_weights.round(6))
q4a_ex_ante = metrics[metrics['portfolio'].isin(q4a_ex_ante_names)].set_index('portfolio')
display(q4a_ex_ante[['expected_return_daily', 'expected_return_annualized', 'volatility_daily', 'ex_ante_beta']].round(6))

q4a_realized = metrics[metrics['portfolio'] == 'beta_hedged_lambda_2'].set_index('portfolio')
display(q4a_realized[['realized_cumulative_return', 'realized_volatility_daily', 'sharpe_annualized', 'realized_beta']].round(6))

## 4(b). Dynamic hedging

In [ ]:
dynamic_weights = weights[weights['portfolio'] == 'q4b_dynamic_beta_hedge_basket']
dynamic_weight_table = dynamic_weights.pivot(index='rebalance_date', columns='asset', values='weight')
display(dynamic_weight_table.round(6))

display(dynamic_comparison.round(6))

## Interpretation of the comparisons

In [ ]:
static = dynamic_comparison['Static Hedge']
dynamic = dynamic_comparison['Dynamic Hedge']
print(f'Dynamic hedge improves cumulative return: {dynamic["Cumulative Return"] > static["Cumulative Return"]}')
print(f'Dynamic hedge reduces volatility: {dynamic["Volatility"] < static["Volatility"]}')
print(f'Dynamic hedge reduces CVaR loss: {dynamic["CVaR 95%"] < static["CVaR 95%"]}')
print(f'Dynamic hedge moves realized beta closer to zero: {abs(dynamic["Realized Beta"]) < abs(static["Realized Beta"])}')

static_ex_ante = q4a_ex_ante.loc['q2b_aggregated_hedged_position', 'expected_return_daily']
return_term_ex_ante = q4a_ex_ante.loc['q2.4a_aggregated_hedged_position', 'expected_return_daily']
print(f'Return-term hedge improves ex-ante expected return: {return_term_ex_ante > static_ex_ante}')